# Task

Network Intrusion Binary Classification 

---

# Prerequisites

Make sure the kernel and environment has the dependencies installed from: `requirements.txt`

In [1]:
import tabm

from pathlib import Path

import pandas as pd
import numpy as np

import scipy.special
import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing

import torch
import torch.nn as nn
import torch.optim
from torch import Tensor

from skimpy import skim

---
# Dataset
> `ToN_IoT`: 
>
> https://www.google.com/url?sa=t&rct=j&q=&esrc=s&source=web&cd=&cad=rja&uact=8&ved=2ahUKEwiQ1e6U7rmXAxW4G4YAHQIeEgcQFnoECA8QAQ&url=https%3A%2F%2Fwww.kaggle.com%2Fdatasets%2Farnobbhowmik%2Fton-iot-network-dataset&usg=AOvVaw1ht079NhkCS_cgIy5cpcYP&opi=89978449

### **Load** the dataset and **summarize**.

In [ ]:
# load and read dataset
data_path = Path('.data/train_test_network.csv')
data = pd.read_csv(data_path)

# prep dataset for summary
data = data.replace('-', pd.NA)

# print summary
skim(data)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 211043 │ │ string      │ 27    │                                                          │
│ │ Number of columns │ 44     │ │ int64       │ 16    │                                                          │
│ └───────────────────┴────────┘ │ float64     │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column_name         ┃ NA ┃ NA % ┃ mean    ┃ sd       ┃ p0 ┃ p25   ┃ p50     ┃ p75    ┃ p100       ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━┩  │
│ │ src_port            │  0 │    0 │   38650 │    19310 │  1 │ 34610 │   44750 │  51130 │      65530 │ ▅  ▇▇▃ │  │
│ │ dst_port            │  0 │    0 │    3495 │    10190 │  0 │    65 │      80 │   1253 │      65470 │   ▇    │  │
│ │ duration            │  0 │    0 │   7.701 │    564.1 │  0 │     0 │ 0.00017 │ 0.0542 │      93520 │   ▇    │  │
│ │ src_bytes           │  0 │    0 │  258100 │ 17090000 │  0 │     0 │       0 │    130 │ 3891000000 │   ▇    │  │
│ │ dst_bytes           │  0 │    0 │  258800 │ 18030000 │  0 │     0 │       0 │     89 │ 3914000000 │   ▇    │  │
│ │ missed_bytes        │  0 │    0 │   34430 │  5262000 │  0 │     0 │       0 │      0 │ 1855000000 │   ▇    │  │
│ │ src_pkts            │  0 │    0 │   9.595 │    91.78 │  0 │     1 │       1 │      4 │      24620 │   ▇    │  │
│ │ src_ip_bytes        │  0 │    0 │   776.1 │    22300 │  0 │    48 │      82 │    415 │    6523000 │   ▇    │  │
│ │ dst_pkts            │  0 │    0 │   3.847 │    330.7 │  0 │     0 │       1 │      2 │     121900 │   ▇    │  │
│ │ dst_ip_bytes        │  0 │    0 │    1585 │   190200 │  0 │     0 │      40 │    134 │   86400000 │   ▇    │  │
│ │ dns_qclass          │  0 │    0 │   227.6 │     2721 │  0 │     0 │       0 │      0 │      32770 │   ▇    │  │
│ │ dns_qtype           │  0 │    0 │   3.611 │     23.8 │  0 │     0 │       0 │      0 │        255 │   ▇    │  │
│ │ dns_rcode           │  0 │    0 │   0.124 │   0.5988 │  0 │     0 │       0 │      0 │          5 │   ▇    │  │
│ │ http_request_body_l │  0 │    0 │ 0.06542 │    9.243 │  0 │     0 │       0 │      0 │       2338 │   ▇    │  │
│ │ en                  │    │      │         │          │    │       │         │        │            │        │  │
│ │ http_response_body_ │  0 │    0 │   144.9 │    30470 │  0 │     0 │       0 │      0 │   13420000 │   ▇    │  │
│ │ len                 │    │      │         │          │    │       │         │        │            │        │  │
│ │ http_status_code    │  0 │    0 │  0.3039 │     8.27 │  0 │     0 │       0 │      0 │        404 │   ▇    │  │
│ │ label               │  0 │    0 │  0.7631 │   0.4252 │  0 │     1 │       1 │      1 │          1 │ ▂    ▇ │  │
│ └─────────────────────┴────┴──────┴─────────┴──────────┴────┴───────┴─────────┴────────┴────────────┴────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━

### **Drop** features with high NA% and **deduplicate** samples.

In [ ]:
# Set the NA% drop threshold. Calculate the NA% for each feature.
na_threshold = 0.80
missing_fraction = data.isna().mean()

# Drop the high NA% features. Make sure to keep the 'label' feature.
high_na_features = missing_fraction[
    (missing_fraction >= na_threshold)
    & ~missing_fraction.index.isin(['label'])
].index.tolist()

data = data.drop(columns=high_na_features)

# Drop 'type' feature since this is binary classification.
data.drop('type')

# Drop any other irrelevant features.

# Deduplicate on all retained columns and reset the row index.
rows_before = len(data)
data = data.drop_duplicates().reset_index(drop=True)

print(f'Dropped {len(high_na_features)} high-NA features: {high_na_features}')
print(f'Removed {rows_before - len(data):,} duplicate rows.')
print(f'Cleaned dataset shape: {data.shape}')
skim(data)


src_ip                    0.000000
src_port                  0.000000
dst_ip                    0.000000
dst_port                  0.000000
proto                     0.000000
service                   0.625617
duration                  0.000000
src_bytes                 0.000000
dst_bytes                 0.000000
conn_state                0.000000
missed_bytes              0.000000
src_pkts                  0.000000
src_ip_bytes              0.000000
dst_pkts                  0.000000
dst_ip_bytes              0.000000
dns_query                 0.834891
dns_qclass                0.000000
dns_qtype                 0.000000
dns_rcode                 0.000000
dns_AA                    0.834095
dns_RD                    0.834095
dns_RA                    0.834095
dns_rejected              0.834095
ssl_version               0.998100
ssl_cipher                0.998100
ssl_resumed               0.998100
ssl_established           0.998100
ssl_subject               0.999948
ssl_issuer          

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 190413 │ │ int64       │ 16    │                                                          │
│ │ Number of columns │ 23     │ │ string      │ 6     │                                                          │
│ └───────────────────┴────────┘ │ float64     │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━┳━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┳━━━━┳━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┓  │
│ ┃ column_name          ┃ NA ┃ NA % ┃ mean    ┃ sd       ┃ p0 ┃ p25   ┃ p50     ┃ p75   ┃ p100       ┃ hist   ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━╇━━━━━━╇━━━━━━━━━╇━━━━━━━━━━╇━━━━╇━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━┩  │
│ │ src_port             │  0 │    0 │   39760 │    18810 │  1 │ 35650 │   45730 │ 52040 │      65530 │ ▃  ▆▇▅ │  │
│ │ dst_port             │  0 │    0 │    2134 │     6967 │  0 │    53 │      80 │   445 │      65470 │   ▇    │  │
│ │ duration             │  0 │    0 │   8.533 │    593.9 │  0 │ 9e-06 │ 0.00039 │ 0.136 │      93520 │   ▇    │  │
│ │ src_bytes            │  0 │    0 │  286100 │ 18000000 │  0 │     0 │       0 │   130 │ 3891000000 │   ▇    │  │
│ │ dst_bytes            │  0 │    0 │  286800 │ 18980000 │  0 │     0 │       0 │   178 │ 3914000000 │   ▇    │  │
│ │ missed_bytes         │  0 │    0 │   38160 │  5539000 │  0 │     0 │       0 │     0 │ 1855000000 │   ▇    │  │
│ │ src_pkts             │  0 │    0 │   10.56 │    96.57 │  0 │     1 │       1 │     5 │      24620 │   ▇    │  │
│ │ src_ip_bytes         │  0 │    0 │   847.8 │    23470 │  0 │    48 │     104 │   423 │    6523000 │   ▇    │  │
│ │ dst_pkts             │  0 │    0 │    4.25 │    348.2 │  0 │     0 │       1 │     2 │     121900 │   ▇    │  │
│ │ dst_ip_bytes         │  0 │    0 │    1756 │   200200 │  0 │     0 │      40 │   234 │   86400000 │   ▇    │  │
│ │ dns_qclass           │  0 │    0 │   251.6 │     2859 │  0 │     0 │       0 │     0 │      32770 │   ▇    │  │
│ │ dns_qtype            │  0 │    0 │   3.913 │    24.92 │  0 │     0 │       0 │     0 │        255 │   ▇    │  │
│ │ dns_rcode            │  0 │    0 │  0.1374 │   0.6289 │  0 │     0 │       0 │     0 │          5 │   ▇    │  │
│ │ http_request_body_le │  0 │    0 │ 0.07251 │    9.731 │  0 │     0 │       0 │     0 │       2338 │   ▇    │  │
│ │ n                    │    │      │         │          │    │       │         │       │            │        │  │
│ │ http_response_body_l │  0 │    0 │   160.5 │    32080 │  0 │     0 │       0 │     0 │   13420000 │   ▇    │  │
│ │ en                   │    │      │         │          │    │       │         │       │            │        │  │
│ │ http_status_code     │  0 │    0 │  0.3274 │    8.597 │  0 │     0 │       0 │     0 │        404 │   ▇    │  │
│ │ label                │  0 │    0 │  0.7795 │   0.4146 │  0 │     1 │       1 │     1 │          1 │ ▂    ▇ │  │
│ └──────────────────────┴────┴──────┴─────────┴──────────┴────┴───────┴─────────┴───────┴────────────┴────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━